<a href="https://colab.research.google.com/github/samanvisingh/samanvi_singh/blob/main/Samanvi_singh_minor_project_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

dim_suppliers = pd.read_csv('dim_suppliers.csv' ,na_values=['N/A', 'missing','--','NA','null'], keep_default_na=True) #
dim_stores = pd.read_csv('dim_stores.csv')
dim_skus = pd.read_csv('dim_skus.csv')
dim_events = pd.read_csv('dim_events.csv')
fact_inventory = pd.read_csv('fact_inventory_daily.csv')

In [4]:
# correcting city casing inconsistency quirk
dim_stores['city_display'] = dim_stores['city_display'].str.title()

In [5]:
df = fact_inventory.merge(dim_stores[['store_id','store_size','sqft']], on='store_id', how='left')
df = df.merge(dim_skus[['sku_id', 'category', 'unit_price_inr','shelf_life_days','is_perishable','festive_relevant']],on='sku_id', how='left')
df = df.merge(dim_suppliers[['supplier_id','reliability_score','base_lead_time_days']], on='supplier_id', how='left')
df = df.merge(dim_events[['date', 'event_name','demand_multiplier_festive','demand_multiplier_other']], on= 'date',how='left')

In [6]:
print("===SANITY VERIFICATION====")
print(f"Total Fact Rows:{len(df)} (Expected:21,600)")
assert len(df) == 21600, "Row count mismatch !"

print("\nTarget class distribution:")
print(df['stockout_risk'].value_counts())

===SANITY VERIFICATION====
Total Fact Rows:21600 (Expected:21,600)

Target class distribution:
stockout_risk
Safe        14131
At-Risk      5186
Imminent     2283
Name: count, dtype: int64


In [14]:
#Reorder gap (distance from trigger point)
df['reorder_gap'] = df['reorder_point'] - df['closing_stock']

 # Normalized days of cover ratio
df['supplier_reliability_score'] = df.groupby('category')['reliability_score'].transform(
    lambda x: x.fillina(x.median())
    )

# Recent reorder signal (rolling 3-day reorder count per store & SKU)
df['date'] = pd.to_datetime(df['date'])
df = df.store_values(['store_id', "sku_id", 'date'])
df['reorder_placed_flag'] = (df['reorder_placed'] == 'Y').astype(int)
df['is_recent_reorder'] = df.groupby(['store_id','sku_id'])['reorder_plaaced_flag'].transform(lambda x: x.shift(1).rolling(window=3,min_period=1).sum()).fillna(0)

df['day_of_month'] = df['date'].dt.day
print(" Feature engineering complete!")

 Feature engineering complete!


In [16]:
# Time based spilt
train_mask = df['date'] <= '2026-10-23'
test_mask = df['date'] >= '2026-10-24'

# Dynamically select features that actually exist in the DataFrame
available_features = [
    col for col in [
        'opening_stock', 'closing_stock', 'sales_velocity_7d', 'days_of_cover',
        'reorder_gap', 'supplier_reliability_score', 'is_recent_reorder', 'day_of_month',
        'category', 'store_size', 'is_perishable', 'festive_relevant'
    ] if col in df.columns
]

target ='stockout_risk'
x_train, y_train = df.loc[train_mask, available_features], df.loc[train_mask, target]
x_test, y_test  = df.loc[test_mask, available_features], df.loc[test_mask, target]
print(f"Using features: {available_features}")
print(f"Training set: {len(x_train)} rows | Testing set :{len(x_test)} rows")

Using features: ['opening_stock', 'closing_stock', 'sales_velocity_7d', 'days_of_cover', 'reorder_gap', 'supplier_reliability_score', 'is_recent_reorder', 'day_of_month', 'category', 'store_size', 'is_perishable', 'festive_relevant']
Training set: 16560 rows | Testing set :5040 rows


In [21]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Dynamically classify columns that are present in x_train
num_cols = [
    col for col in [
        'opening_stock', 'closing_stock', 'sales_velocity_7d', 'days_of_cover',
        'reorder_gap', 'supplier_reliability_score', 'is_recent_reorder', 'day_of_month'
    ] if col in x_train.columns
]

cat_cols = [
    col for col in ['category', 'store_size', 'is_perishable', 'festive_relevant']
    if col in x_train.columns
]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

#Build Random Forest Pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

#fit model
model_pipeline.fit(x_train, y_train)

# evaluate prediction
y_pred = model_pipeline.predict(x_test)

print("==== MODEL PERFORMANCE REPORT ====")
print("Overall Accuracy:", round(accuracy_score(y_test, y_pred),4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred, labels=['Safe', 'At-Risk', 'Imminent']))

==== MODEL PERFORMANCE REPORT ====
Overall Accuracy: 0.6444

Classification Report:
               precision    recall  f1-score   support

     At-Risk       0.25      0.02      0.03      1198
    Imminent       0.25      0.00      0.01       561
        Safe       0.65      0.98      0.78      3281

    accuracy                           0.64      5040
   macro avg       0.38      0.33      0.27      5040
weighted avg       0.51      0.64      0.52      5040

Confusion Matrix:
 [[3225   52    4]
 [1175   21    2]
 [ 549   10    2]]
